# 🎥 Nexus Cloud Video Renderer (Headless Canvas to MP4)
Este notebook realiza a renderização automatizada do documentário animado completo `/ensaio` de 12.2 minutos utilizando aceleração por hardware GPU na nuvem.

In [ ]:
# @title 1. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. Instalar Dependências do Sistema (Node, FFmpeg, Vulkan)
print("🚀 Instalando dependências de sistema...")
!apt-get update -qq
!apt-get install -qq -y build-essential ffmpeg libvulkan-dev libwayland-dev libx11-dev nodejs npm
print("✅ Dependências instaladas!")

In [ ]:
# @title 3. Clonar e Configurar o Repositório do Vídeo
import os
PROJECT_DIR = "/content/nexus-video-ensaio"
if not os.path.exists(PROJECT_DIR):
  print("🛸 Clonando repositório de vídeo...")
  !git clone -b master https://github.com/Mateus1746/nexus-video-ensaio.git {PROJECT_DIR}
else:
  print("✅ Repositório já clonado. Atualizando para a branch mais recente...")
  !cd {PROJECT_DIR} && git checkout master && git pull origin master

print("📦 Instalando dependências do gravador headless...")
!cd {PROJECT_DIR}/tools/Engine-Headless-Recorder && npm install

print("📦 Instalando dependências do projeto web...")
!cd {PROJECT_DIR} && npm install
print("✅ Projeto configurado com sucesso!")

In [ ]:
# @title 4. Sincronizar Arquivos SOTA (Garantir narrativa e áudios mais recentes)
# Copia a narrativa sincronizada de 12 minutos do Drive para o repositório
!cp /content/drive/MyDrive/nexus_pipeline/staging/ensaio/pipeline/narrative.json /content/nexus-video-ensaio/pipeline/narrative.json
print("✅ narrative.json sincronizado com sucesso!")

In [ ]:
# @title 5. Iniciar Renderização Headless do Documentário Completo (1080p GPU)
import os
import subprocess

# Configurações do driver gráfico Vulkan Headless do Colab
os.environ["WGPU_BACKEND"] = "vulkan"
os.environ["WGPU_POWER_PREFERENCE"] = "high"
os.environ["DISPLAY"] = ""

OUTPUT_DIR = "/content/drive/MyDrive/nexus_pipeline/audio_ready/ensaio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🎬 Executando Gravação Headless do Canvas...")
recorder_path = "/content/nexus-video-ensaio/tools/Engine-Headless-Recorder/src/node/record_video.js"
output_file = os.path.join(OUTPUT_DIR, "cold_war_documentary_final.mp4")

# Comando executando a gravação com o Chrome Headless em modo GPU acelerado
cmd = [
    "node", recorder_path,
    "--canvas=#video-canvas",
    "--duration=737.53",
    "--fps=25",
    f"--output={output_file}",
    "--mode=gpu"
]

print(f"🚀 Executando: {' '.join(cmd)}")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode == 0:
    print(f"\n🎉 SUCESSO! O vídeo animado real foi gravado e salvo em: {output_file}")
else:
    print(f"\n❌ Erro na renderização. Código: {process.returncode}")